# Dynamic views: Securing data at the row level using Databricks Unity Catalog

**Note: using dynamic view was the solution before Row level and column-level masking with SQL FUNCTIONS**

**We recommend using the previous [01-Row-Column-access-control]($./01-Row-Column-access-control) notebook over adding dynamic views when possible.**

As seen in the previous notebook, Unity Catalog let you grant table ACL using standard SQL GRANT on all the objects (CATALOG, SCHEMA, TABLE)

But this alone isn't enough. UC let you create more advanced access pattern to dynamically filter your data based on who query it.

This is usefull to mask sensitive PII information, or restrict access to a subset of data without having to create and maintain multiple tables.

*Note that Unity Catalog will provide more advanced data masking capabilities in the future, this demo covers what can be done now.*

*Note: This is currently only supported with shared cluster (python/SQL). Single node requires access to the underlying view*

See the [documentation](https://docs.databricks.com/security/access-control/table-acls/object-privileges.html#dynamic-view-functions) for more details.

<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=governance&org_id=7474650537677415&notebook=%2F02-%5Blegacy%5D-UC-Dynamic-view&demo_name=uc-01-acl&event=VIEW&path=%2F_dbdemos%2Fgovernance%2Fuc-01-acl%2F02-%5Blegacy%5D-UC-Dynamic-view&version=1">

## Cluster setup for UC

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/uc/clusters_shared.png?raw=true" width="600" style="float: right"/>

To be able to run this demo, make sure you create a cluster with the security mode enabled.

1. Go in the compute page, create a new cluster

2. Under "Access mode", select "Shared"

In [0]:
%run ./_resources/00-setup

USE CATALOG `main_build`
using catalog.database `main_build`.`dbdemos_uc_01_acl`


## Current user and is member (group)

Databricks has 2 functions: `current_user()` and `is_account_group_member()`.

Theses functions can be used to dynamically get the user running the query and knowing if the user is member of a give group.

In [0]:
SELECT current_user();

current_user()
quentin.ambard@databricks.com


In [0]:
-- Note: The account should have been setup by adding all users to the ANALYST_USA group
SELECT is_account_group_member('account users'), is_account_group_member('ANALYST_USA'), is_account_group_member('ANALYST_FR');

is_account_group_member(account users),is_account_group_member(ANALYST_USA),is_account_group_member(ANALYST_FR)
true,false,false


## Dynamic Views: Restricting data to a subset based on a field

We'll be using the previous customers table. Let's review it's content first.

*Note: Make sure you run the [previous notebook]($00-UC-Table-ACL) first*

In [0]:
SELECT * FROM customers

id,creation_date,firstname,lastname,country,email,address,gender,age_group,canal,last_activity_date,churn
f120372e-022b-41b6-88f7-d6aec8565a6e,04-19-2013 00:00:00,Christopher,Davis,SPAIN,sbrooks@hebert.com,"0866 Luna Crest Briggsside, AR 02197",1.0,3.0,WEBAPP,06-04-2023 16:55:26,true
68eac5be-ba02-4b28-898b-f531b0e294f4,01-21-2014 00:00:00,Andrea,Pierce,USA,ybriggs@pierce.org,"4600 Richardson Dale New Wanda, NE 42581",1.0,2.0,MOBILE,06-06-2023 20:54:21,true
e4ac5e22-4112-4ade-be03-ebc9a4fcd971,08-05-2021 00:00:00,Jane,Padilla,USA,sellersmichael@mitchell.com,"286 Angela Row South Rachel, AL 90762",1.0,6.0,PHONE,06-02-2023 20:30:20,true
85b5cb37-703e-4b0b-9fd2-e26bfc28ebd6,07-22-2021 00:00:00,Deanna,Brewer,SPAIN,wnunez@cole.com,"15141 Mendoza Brook Port Richardhaven, NE 17171",0.0,6.0,MOBILE,06-01-2023 17:08:48,false
ae08a336-3126-4ef8-b8c0-2ca6b0277593,08-02-2021 00:00:00,Carla,Harper,FR,traceyhoward@zhang-gutierrez.org,"3583 Green Squares Port Jameshaven, MO 58447",1.0,2.0,MOBILE,06-04-2023 21:36:07,true
449a2bba-a29d-48dc-b623-2acd83dea4fb,07-24-2021 00:00:00,Joel,Young,FR,cathy85@silva.com,"16999 Michael Alley Apt. 910 Nguyenport, KS 99031",1.0,6.0,WEBAPP,06-02-2023 21:02:46,true
db8777a9-c7d8-4cae-9bac-80f4afd269b8,08-14-2021 00:00:00,Ian,White,USA,marie14@patrick-deleon.com,"123 Christian Village Suite 413 Port Ashleyport, VA 44207",1.0,4.0,WEBAPP,06-04-2023 16:39:48,false
7339fbd2-5b2b-4b9a-82c6-dc577ef0cf91,07-24-2021 00:00:00,Elizabeth,Ortega,FR,marshstacy@serrano.info,"58280 Brittany Terrace Apt. 046 Lake Pamelatown, PA 72400",1.0,6.0,PHONE,06-08-2023 08:04:54,false
65ae08c1-4327-47ce-b1e7-f88b196bf015,08-06-2021 00:00:00,Dana,Walsh,FR,veronica63@cardenas.com,Unit 4287 Box 8773 DPO AE 21291,1.0,2.0,MOBILE,06-04-2023 13:00:52,true
23671c7d-06b2-4a0d-a750-8faefd8348fe,07-23-2021 00:00:00,James,Harris,SPAIN,aaronbrooks@combs.info,"57462 Duncan Land Stephenston, ND 98930",1.0,8.0,WEBAPP,06-07-2023 03:58:15,true


As you can see, this table has a `country`field. We want to be able to restrict the table access based in this country.

Data Analyst and Data Scientists in USA can only access the local Dataset, same for the FR team.

### Using groups
One option to do that would be to create groups in the Unity Catalog. We can name the groups as the concatenation of `CONCAT("ANALYST_", country)`:
* `ANALYST_FR`
* `ANALYST_USA`. 
* `ANALYST_SPAIN`. 

You can then add a view with `CASE ... WHEN` statement based on your groups to define when the data can be accessed.

See the [documentation](https://docs.databricks.com/security/access-control/table-acls/object-privileges.html#dynamic-view-functions) for more details on that.

But what makes the `is_member()` function powerful is that you can combine it with a column. Let's see how we can use it to dynamically check access based on the row.
 
We'll create a field named `group_name` as the concatenation of ANALYST and the country, and then for each value check if the current user is a member of this group:

In [0]:
-- as ANALYST from the USA (ANALYST_USA group), each USA row are now at "true"
SELECT is_account_group_member(group_name), * FROM (
  SELECT CONCAT("ANALYST_", country) AS group_name, country, id, firstname FROM customers)

is_account_group_member(group_name),group_name,country,id,firstname
false,ANALYST_USA,USA,c0586d5e-66a9-43bd-bd45-d486471dc184,Jennifer
false,ANALYST_USA,USA,63bc9985-e4fe-448b-8f09-ac7664009b12,Jeffrey
false,ANALYST_SPAIN,SPAIN,572cbc8b-8b60-4da0-be10-0df5bc64a022,Kari
false,ANALYST_FR,FR,cf5399a1-f54c-4a1c-9b56-0954f4d95c67,Christopher
false,ANALYST_SPAIN,SPAIN,054bf38b-4dcd-4182-8fde-b6b7d8c8b440,Jessica
false,ANALYST_USA,USA,9e3abb7c-198a-4b81-b71d-3378b09cc00d,Susan
false,ANALYST_SPAIN,SPAIN,02b2815e-eafb-4f03-aee2-49b77b4fb557,Joseph
false,ANALYST_USA,USA,ae016967-ce91-4242-8645-eb7bdbd46805,Tina
false,ANALYST_FR,FR,8fd9ad1c-79d0-4acc-952c-d77f392c8cce,Daniel
false,ANALYST_FR,FR,bc51d46f-0033-4ba1-8d99-bc2a4da5eec8,Aaron


As you can see, we are not admin on any of these group.
We can create a view securiting this data and only grant our analyst access to this view: 

In [0]:
CREATE OR REPLACE VIEW customer_dynamic_view  AS (
  SELECT * FROM customers as customers WHERE is_account_group_member(CONCAT("ANALYST_", country))
);
-- Then grant select access on the view only
GRANT SELECT ON VIEW customer_dynamic_view TO `account users`;

Because we're not part of any group, we won't have access to the data. Users being in the `ANALYST_FR` group will have a filter to access only the FR country.

All we have to do now is add our users to the groups to be able to have access

In [0]:
-- We should be part of the ANALYST_USA group. As result, we now have a row-level filter applied in our secured view and we only see the USA country:
select * from customer_dynamic_view

id,creation_date,firstname,lastname,country,email,address,gender,age_group,canal,last_activity_date,churn


## Dynamic Views & data masking

The country example was a first level of row-level security implementation. We can implement more advances features using the same pattern.

Let's see how Dynamic views can also be used to add data masking. For this example we'll be using the `current_user()` functions.

Let's create a table with all our current analyst permission including a GDPR permission flag: `analyst_permissions`.

This table has 3 field:

* `analyst_email`: to identify the analyst (we could work with groups instead)
* `country_filter`: we'll filter the dataset based on this value
* `gdpr_filter`: if true, we'll filter the PII information from the table. If not set the user can see all the information

*Of course this could be implemented with the previous `is_account_group_member()` function instead of individual users information being saved in a permission tale.*

Let's query this table and check our current user permissions. As you can see I don't have GDPR filter enabled and a filter on FR is applied for me in the permission table we created.

In [0]:
select * from analyst_permissions where analyst_email = current_user()

analyst_email,country_filter,gdpr_filter
quentin.ambard@databricks.com,USA,1


In [0]:
CREATE OR REPLACE VIEW customer_dynamic_view_gdpr AS (
  SELECT 
  id ,
  creation_date,
  country,
  gender,
  age_group,
  CASE WHEN country.gdpr_filter=1 THEN sha1(firstname) ELSE firstname END AS firstname,
  CASE WHEN country.gdpr_filter=1 THEN sha1(lastname)  ELSE lastname  END AS lastname,
  CASE WHEN country.gdpr_filter=1 THEN sha1(email)     ELSE email     END AS email
  FROM 
    customers as customers INNER JOIN 
    analyst_permissions country  ON country_filter=country
  WHERE 
    country.analyst_email=current_user() 
);
-- Then grant select access on the view only
GRANT SELECT ON VIEW customer_dynamic_view_gdpr TO `account users`;


## Querying the secured view
Let's now query the view. Because I've a filter on `COUNTRY=FR`and `gdpr_filter=0`, I'll see all the FR customers information. 

In [0]:
%sql select * from customer_dynamic_view_gdpr 

id,creation_date,country,gender,age_group,firstname,lastname,email
c0586d5e-66a9-43bd-bd45-d486471dc184,09-14-2022 00:00:00,USA,1.0,5.0,7f0871085cb3a34c4b02428e49b07cd77e0231f4,649c0affc28554f08b5243d89394ab61d39cdcc9,d53534bf82b458183fba816d6bd032dfa023d1e3
63bc9985-e4fe-448b-8f09-ac7664009b12,10-10-2022 00:00:00,USA,0.0,9.0,72886a8e7d21c637750593e98faf71cf54e683d1,a95075f5042b1b27060080156d87fe34ec7e712c,981c61452bdf681a012a32ed15b870ddc7a0c762
9e3abb7c-198a-4b81-b71d-3378b09cc00d,09-17-2022 00:00:00,USA,1.0,6.0,33dca1c0c1c5ae78f67580a76d9c6aba6a172e20,e77d4d8057e8d99379dd691fb53a79a0971753cc,5878a06a0a42dfd4d3efc824f40b85dc915b1dc4
ae016967-ce91-4242-8645-eb7bdbd46805,09-13-2022 00:00:00,USA,0.0,0.0,79e1f69e118a4c2a04f4e0bef79bd8d3feb61d69,0877a1240b9b24dc29e5a58c275db5a72d011ce8,8a84fc9662600a94a854ae8d30e93204bf9ab832
757ddae0-2dbe-4509-a9c0-fb92b39c6c01,10-09-2022 00:00:00,USA,0.0,4.0,5d569dfc13001c8b30aa11eeb2a59d22071f3d80,6032a672e309a4dfd10a6c41199d5fd8c5c83da9,4f19808ad3bc6a13e6082ae52ee611bf8c0cbe53
b688620d-4958-43c4-b677-625bd07140de,10-27-2022 00:00:00,USA,1.0,5.0,f941e1206abd4a2d8889da67be10151f429d95dc,96bcf8c98f94b6ace4a4b716cf0e3b32743a08b1,62926b8de4b8390bd4a1e44f2f36a0f65e20d6d1
dc89af81-54d5-4bf6-9063-9b3e5a08eb0b,10-20-2022 00:00:00,USA,1.0,4.0,6b15b34bf98d4b78fadf01a78cc201c927d6f452,dabe73052e584c5d5ea0dae5b7ef23f36a979331,0a5ceffa61fc87c9964713ab694330f1bbe5c12a
92811796-5630-4b6b-a6c5-bef1dd2af700,10-22-2022 00:00:00,USA,0.0,3.0,7404a4c6a588027455d739c03238b869c00a8d52,8511af2257d1274d0629d8f005851ae2715251b5,611fc520326dfd7c5570b251a97cc52859d1f8f8
216113a3-1511-483d-8e27-e0468b1f342a,10-19-2022 00:00:00,USA,0.0,1.0,b6421c86686c7d176e682f87db1e3a27a55cb877,b2c07102495cfbdd0e898d63e778b0ea5ab4f4f5,07bf4252f5b0aeed7717f4f5bd62dfac9eb89498
117e4909-4896-4d7a-b76a-bba2dee94edd,10-27-2022 00:00:00,USA,1.0,4.0,4e62fdd419fb6451658fd5b70f558f95b20d416d,c4633c096745cff1cd1cf948344963fbfff4356c,0b7a9e799fcb6b7809dec8ea158fa46627258c7d


Let's now change my permission. We'll enable the `gdpr_filter` flag and change our `country_filter` to USA.

As you can see, requesting the same secured view now returns all the USA customers, and PII information has been obfuscated:

In [0]:
UPDATE analyst_permissions SET country_filter='USA', gdpr_filter=1 where analyst_email=current_user();

select * from customer_dynamic_view_gdpr ;

id,creation_date,country,gender,age_group,firstname,lastname,email
c0586d5e-66a9-43bd-bd45-d486471dc184,09-14-2022 00:00:00,USA,1.0,5.0,7f0871085cb3a34c4b02428e49b07cd77e0231f4,649c0affc28554f08b5243d89394ab61d39cdcc9,d53534bf82b458183fba816d6bd032dfa023d1e3
63bc9985-e4fe-448b-8f09-ac7664009b12,10-10-2022 00:00:00,USA,0.0,9.0,72886a8e7d21c637750593e98faf71cf54e683d1,a95075f5042b1b27060080156d87fe34ec7e712c,981c61452bdf681a012a32ed15b870ddc7a0c762
9e3abb7c-198a-4b81-b71d-3378b09cc00d,09-17-2022 00:00:00,USA,1.0,6.0,33dca1c0c1c5ae78f67580a76d9c6aba6a172e20,e77d4d8057e8d99379dd691fb53a79a0971753cc,5878a06a0a42dfd4d3efc824f40b85dc915b1dc4
ae016967-ce91-4242-8645-eb7bdbd46805,09-13-2022 00:00:00,USA,0.0,0.0,79e1f69e118a4c2a04f4e0bef79bd8d3feb61d69,0877a1240b9b24dc29e5a58c275db5a72d011ce8,8a84fc9662600a94a854ae8d30e93204bf9ab832
757ddae0-2dbe-4509-a9c0-fb92b39c6c01,10-09-2022 00:00:00,USA,0.0,4.0,5d569dfc13001c8b30aa11eeb2a59d22071f3d80,6032a672e309a4dfd10a6c41199d5fd8c5c83da9,4f19808ad3bc6a13e6082ae52ee611bf8c0cbe53
b688620d-4958-43c4-b677-625bd07140de,10-27-2022 00:00:00,USA,1.0,5.0,f941e1206abd4a2d8889da67be10151f429d95dc,96bcf8c98f94b6ace4a4b716cf0e3b32743a08b1,62926b8de4b8390bd4a1e44f2f36a0f65e20d6d1
dc89af81-54d5-4bf6-9063-9b3e5a08eb0b,10-20-2022 00:00:00,USA,1.0,4.0,6b15b34bf98d4b78fadf01a78cc201c927d6f452,dabe73052e584c5d5ea0dae5b7ef23f36a979331,0a5ceffa61fc87c9964713ab694330f1bbe5c12a
92811796-5630-4b6b-a6c5-bef1dd2af700,10-22-2022 00:00:00,USA,0.0,3.0,7404a4c6a588027455d739c03238b869c00a8d52,8511af2257d1274d0629d8f005851ae2715251b5,611fc520326dfd7c5570b251a97cc52859d1f8f8
216113a3-1511-483d-8e27-e0468b1f342a,10-19-2022 00:00:00,USA,0.0,1.0,b6421c86686c7d176e682f87db1e3a27a55cb877,b2c07102495cfbdd0e898d63e778b0ea5ab4f4f5,07bf4252f5b0aeed7717f4f5bd62dfac9eb89498
117e4909-4896-4d7a-b76a-bba2dee94edd,10-27-2022 00:00:00,USA,1.0,4.0,4e62fdd419fb6451658fd5b70f558f95b20d416d,c4633c096745cff1cd1cf948344963fbfff4356c,0b7a9e799fcb6b7809dec8ea158fa46627258c7d


## Conclusion

As we've seen, data masking and filtering can be implemented at a row level using groups, users and even extra table that you can use to manage more advanced permissions.

You're now ready to deploy the Lakehouse for your entire organisation, securing data based on your own governance, ensuring PII regulation and governance.